# Phase 2 — Real Solar & Meteorological Data Collection (Settat, Morocco)

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Provisional Site:** Settat, Casablanca-Settat, Morocco (`33.0010° N`, `-7.6166° E`)

## Learning Objectives
1. Retrieve historical hourly **solar irradiance & reference PV output** from **EC PVGIS v5.3** (`seriescalc`).
2. Retrieve historical hourly **ambient temperature, relative humidity, and wind speed** from **Open-Meteo ERA5**.
3. Preserve raw downloads as **immutable archives** in `data/raw/` with SHA-256 verification.
4. Standardize timestamps to **UTC**, validate physical ranges **without synthetic imputation**, and export clean chronological splits (`2020–2021`, `2022`, `2023`).


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'requests>=2.31.0'])

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'src/data_collection.py',
    'src/preprocessing.py',
    'data/processed/solar_weather_merged_clean.csv',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PROCESSED_DATA_DIR, RAW_DATA_DIR
from src.data_collection import collect_all_phase2_data
from src.preprocessing import preprocess_and_split_phase2

sns.set_theme(style='whitegrid')

# Uses cached immutable raw data if already present; downloads from APIs cleanly in Colab if missing
metadata = collect_all_phase2_data(overwrite=False)
print('Site       :', metadata['provisional_site']['site_name'])
print('PVGIS rows :', metadata['sources']['pvgis']['record_count'])
print('Meteo rows :', metadata['sources']['open_meteo']['record_count'])

quality_report = preprocess_and_split_phase2()
print('Merged rows:', quality_report['merged_rows'], '| Missing:', quality_report['missing_hours'], '| Duplicates:', quality_report['duplicate_timestamps'])


In [ ]:
# Visual Inspection of Real Solar & Weather Data + Export Tabular Quality Summary
df = pd.read_csv(PROCESSED_DATA_DIR / 'solar_weather_merged_clean.csv', parse_dates=['timestamp'])
assert df['timestamp'].is_monotonic_increasing, 'Timestamps must be strictly chronological!'
assert int(df.isna().sum().sum()) == 0, 'Unexpected missing values!'

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
diurnal = df.groupby('hour_utc')[['pvgis_irradiance_wm2', 'pvgis_pv_power_kw']].mean()
axes[0].plot(diurnal.index, diurnal['pvgis_irradiance_wm2'], color='#D97706', lw=2)
axes[0].set_title('Settat Mean Diurnal Solar Irradiance (2020–2023 UTC)')
axes[0].set_xlabel('Hour (UTC)'); axes[0].set_ylabel('W/m²')

monthly = df.groupby('month')[['meteo_ambient_temp_c', 'meteo_relative_humidity_pct']].mean()
axes[1].plot(monthly.index, monthly['meteo_ambient_temp_c'], marker='o', color='#0284C7', label='Temp (°C)')
axes[1].set_title('Settat Monthly Ambient Temperature (ERA5)')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Temperature (°C)')
plt.tight_layout(); plt.show()


## Conclusions & Limitations
- All `35,064` UTC hours across `2020–2023` align cleanly (`0` missing hours, `0` duplicates, `0` synthetic imputations).
- PV output (`pvgis_pv_power_kw`) reflects a PVGIS `15.0 kWp` reference system and **not** measured pilot-plant telemetry.


In [ ]:
# Export & Download Phase 2 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_02'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase02_Package.zip'

patterns = ['src/config.py', 'src/data_collection.py', 'src/preprocessing.py', 'docs/phase2_data_provenance.md', 'data/raw/*.csv', 'data/raw/*.json', 'data/processed/*.csv', 'data/processed/*.json', 'notebooks/01_data_collection.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
